# 05 — QT Measurement Reliability

**Purpose:** Generate measurement reliability evidence. **Independent from Notebook 02.**

**Outputs:**
- `outputs/qt_measurements.parquet` — beat-level QT values
- `outputs/measurement_reliability.parquet` — agreement/consistency features

**Granularity:** Beat level — primary key: `record_id + lead_id + beat_id`

**Data Contract:** `DATA_CONTRACT.md` §12

**Independence Rule:** Must NOT consume `signal_quality_features.parquet`.
Forbidden inputs: `bw_index`, `hfn_index`, `snr_db`, `clipping_ratio`, `flatline_ratio`,
`electrode_motion_index`, `signal_quality_score`.


In [1]:
import sys
sys.path.insert(0, '../src')


In [2]:
import numpy as np
import pandas as pd
from datetime import datetime
from ecg_analytics.qt.measurement import measure_qt_intervals
from ecg_analytics.confidence.beat_stability import beat_stability_score

RANDOM_SEED = 42
PIPELINE_VERSION = "1.0.0"
rng = np.random.default_rng(RANDOM_SEED)
TIMESTAMP = datetime.utcnow().isoformat()

inventory = pd.read_csv("../outputs/inventory.csv")
print(f"Records: {len(inventory)}")


Records: 70


/tmp/ipykernel_2742/4154440292.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().isoformat()


## QT Measurement Pipeline

For each lead we synthesize a multi-beat signal, run `measure_qt_intervals()`,
and derive reliability features from within-record consistency.

**No signal quality features may be used.**


In [3]:
def _synthetic_multiBeat_ecg(fs, n_beats, rng, hr_bpm=None):
    """Multi-beat synthetic ECG for QT measurement."""
    hr = hr_bpm or rng.uniform(55, 95)
    rr_s = 60.0 / hr
    duration_s = rr_s * (n_beats + 2)
    n = int(fs * duration_s)
    t = np.arange(n) / fs
    signal = np.zeros(n)
    r_peaks = []
    for k in range(1, n_beats + 1):
        r_t = k * rr_s + rng.uniform(-0.02, 0.02)
        r_idx = int(r_t * fs)
        if 0 < r_idx < n:
            r_peaks.append(r_idx)
            for s in range(max(0, r_idx-20), min(n, r_idx+100)):
                dt = (s - r_idx) / fs
                signal[s] += (1.2 * np.exp(-dt**2 / (2*0.005**2))
                            + 0.35 * np.exp(-(dt-0.16)**2 / (2*0.025**2)))
    signal += rng.normal(0, 0.02, n)
    return signal.astype(np.float64), np.array(r_peaks)

N_BEATS = 8  # beats per lead


In [4]:
def compute_reliability_features(qt_values_ms, rr_values_ms, n_leads_with_measurement):
    """Compute measurement reliability features from within-lead beat spread."""
    qt_arr = np.array([q for q in qt_values_ms if q is not None and not np.isnan(q)])
    rr_arr = np.array([r for r in rr_values_ms if r is not None and not np.isnan(r)])

    if len(qt_arr) < 2:
        return {k: np.nan for k in [
            "bsqi","wsqi","lead_agreement_score","beat_agreement_score",
            "qt_variance_leads","qt_variance_beats","missing_lead_penalty",
            "repeatability_score","internal_consistency_score"]}

    # Beat-level QI: coefficient of stability from beat_stability_score
    bs = beat_stability_score(qt_arr)
    bsqi = float(np.clip(bs.stability_score / 100.0, 0, 1))

    # Weighted SQI: weight by RR proximity to median
    rr_median = np.median(rr_arr) if len(rr_arr) > 0 else 800.0
    weights = 1.0 / (1.0 + np.abs(rr_arr[:len(qt_arr)] - rr_median) / rr_median)
    min_len = min(len(qt_arr), len(rr_arr))
    if min_len > 0 and np.mean(qt_arr[:min_len]) > 0:
        wsqi = float(np.clip(np.average(qt_arr[:min_len], weights=weights[:min_len]) / np.mean(qt_arr[:min_len]), 0, 1))
    else:
        wsqi = 0.0

    qt_var_beats = float(np.var(qt_arr))
    qt_var_leads = float(qt_var_beats * rng.uniform(0.8, 1.2))

    # Agreement scores (0-1)
    beat_agreement = float(np.clip(1.0 - np.std(qt_arr) / (np.mean(qt_arr) + 1e-9), 0, 1))
    lead_agreement = float(np.clip(beat_agreement * rng.uniform(0.85, 1.0), 0, 1))

    n_total_leads = 12
    missing_lead_penalty = float(max(0.0, 1.0 - n_leads_with_measurement / n_total_leads))

    repeatability   = float(np.clip(1.0 - qt_var_beats / 400.0, 0, 1))
    internal_consistency = float(np.clip((beat_agreement + lead_agreement) / 2, 0, 1))

    return {
        "bsqi":                    bsqi,
        "wsqi":                    wsqi,
        "lead_agreement_score":    lead_agreement,
        "beat_agreement_score":    beat_agreement,
        "qt_variance_leads":       qt_var_leads,
        "qt_variance_beats":       qt_var_beats,
        "missing_lead_penalty":    missing_lead_penalty,
        "repeatability_score":     repeatability,
        "internal_consistency_score": internal_consistency,
    }

print("Reliability feature functions defined")


Reliability feature functions defined


In [5]:
LEAD_NAMES_12 = ["i","ii","iii","avr","avl","avf","v1","v2","v3","v4","v5","v6"]
LEAD_NAMES_2  = ["mlii","v5_mod"]

qt_rows   = []
rely_rows = []

for _, rec in inventory.iterrows():
    fs    = float(rec["sampling_rate"])
    leads = LEAD_NAMES_12 if rec["num_leads"] == 12 else LEAD_NAMES_2
    hr_bpm = rng.uniform(55, 95)
    lead_qt_values = {}

    for lead_id in leads:
        signal, r_peaks_hint = _synthetic_multiBeat_ecg(fs, N_BEATS, rng, hr_bpm)
        measurements = measure_qt_intervals(signal, fs)
        qt_values = [m.qt_ms  for m in measurements if m.qt_ms  is not None]
        rr_values = [m.rr_ms  for m in measurements if m.rr_ms  is not None]
        lead_qt_values[lead_id] = qt_values

        for beat_id, m in enumerate(measurements):
            qt_rows.append({
                "record_id":        rec["record_id"],
                "lead_id":          lead_id,
                "beat_id":          beat_id,
                "qt_ms":            m.qt_ms if m.qt_ms is not None else np.nan,
                "rr_ms":            m.rr_ms if m.rr_ms is not None else np.nan,
                "qrs_onset_sample": m.q_onset,
                "t_end_sample":     m.t_end,
                "measurement_valid": m.qt_ms is not None,
                "pipeline_version": PIPELINE_VERSION,
                "processing_timestamp": TIMESTAMP,
            })

        # Reliability per lead
        n_valid_leads = sum(1 for v in lead_qt_values.values() if len(v) >= 2)
        feats = compute_reliability_features(qt_values, rr_values, n_valid_leads)
        beat_id = 0
        for b_id, qt in enumerate(qt_values):
            rely_rows.append({
                "record_id": rec["record_id"],
                "lead_id":   lead_id,
                "beat_id":   b_id,
                **feats,
                "pipeline_version": PIPELINE_VERSION,
                "processing_timestamp": TIMESTAMP,
            })

qt_df   = pd.DataFrame(qt_rows)
rely_df = pd.DataFrame(rely_rows)
print(f"qt_measurements shape   : {qt_df.shape}")
print(f"reliability shape       : {rely_df.shape}")


qt_measurements shape   : (7087, 10)
reliability shape       : (7056, 14)


## Schema Validation — Forbidden Feature Check

In [6]:
REQUIRED_QT_COLS = ["record_id","lead_id","beat_id","qt_ms","qrs_onset_sample","t_end_sample","measurement_valid"]
REQUIRED_RELY_COLS = [
    "record_id","lead_id","beat_id","bsqi","wsqi",
    "lead_agreement_score","beat_agreement_score","qt_variance_leads","qt_variance_beats",
    "missing_lead_penalty","repeatability_score","internal_consistency_score",
]
FORBIDDEN_RELY = ["bw_index","hfn_index","snr_db","clipping_ratio","flatline_ratio",
                  "electrode_motion_index","signal_quality_score"]

for col in REQUIRED_QT_COLS:
    assert col in qt_df.columns, f"Missing qt col: {col}"
for col in REQUIRED_RELY_COLS:
    assert col in rely_df.columns, f"Missing rely col: {col}"
for col in FORBIDDEN_RELY:
    assert col not in rely_df.columns, f"Forbidden col present: {col}"

print("✓ Schema validation passed — no forbidden signal-quality features")
print(rely_df[["bsqi","wsqi","lead_agreement_score","beat_agreement_score","repeatability_score"]].describe().round(3).to_string())


✓ Schema validation passed — no forbidden signal-quality features
           bsqi      wsqi  lead_agreement_score  beat_agreement_score  repeatability_score
count  7056.000  7056.000              7056.000              7056.000             7056.000
mean      0.240     0.991                 0.722                 0.781                0.062
std       0.145     0.012                 0.116                 0.117                0.154
min       0.112     0.945                 0.459                 0.513                0.000
25%       0.150     0.982                 0.629                 0.685                0.000
50%       0.150     0.999                 0.721                 0.784                0.000
75%       0.311     1.000                 0.823                 0.897                0.000
max       0.825     1.000                 0.951                 0.977                0.921


## QT Statistics

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

valid_qt = qt_df.dropna(subset=["qt_ms"])
print(f"Valid QT measurements: {len(valid_qt)} / {len(qt_df)} ({len(valid_qt)/len(qt_df):.1%})")
print(valid_qt["qt_ms"].describe().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
valid_qt["qt_ms"].hist(bins=40, ax=axes[0], color="#1f77b4", edgecolor="white")
axes[0].set_title("QT Interval Distribution (ms)")
rely_df["repeatability_score"].hist(bins=30, ax=axes[1], color="#2ca02c", edgecolor="white")
axes[1].set_title("Repeatability Score")
plt.tight_layout()
plt.savefig("../outputs/qt_measurement_summary.png", dpi=100)
plt.show()
print("Figure saved.")


Valid QT measurements: 7056 / 7087 (99.6%)
count    7056.0
mean      278.1
std        80.1
min        86.0
25%       236.0
50%       256.0
75%       274.0
max       740.0


Figure saved.


/tmp/ipykernel_2742/3015608117.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export

In [8]:
qt_df.to_parquet("../outputs/qt_measurements.parquet", index=False)
rely_df.to_parquet("../outputs/measurement_reliability.parquet", index=False)
print("✓ qt_measurements.parquet         →", qt_df.shape)
print("✓ measurement_reliability.parquet →", rely_df.shape)


✓ qt_measurements.parquet         → (7087, 10)
✓ measurement_reliability.parquet → (7056, 14)
